# **ST 554 Homework 10**
---
Authored by: Jamie Loring

Collaborator: Dr. Justin Post (code from lecture videos & slides)

In [1]:
#importing required modules
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, sqrt
from pyspark.ml.feature import SQLTransformer, VectorAssembler
from pyspark.ml import Pipeline

The code below creates a spark object for use with this entire assignment.

In [2]:
spark = SparkSession.builder.getOrCreate()

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/04/20 23:22:13 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
26/04/20 23:22:13 WARN Utils: Service 'SparkUI' could not bind on port 4040. Attempting port 4041.
26/04/20 23:22:13 WARN Utils: Service 'SparkUI' could not bind on port 4041. Attempting port 4042.


The code below only allows error messages to print out going forward.

In [3]:
spark.sparkContext.setLogLevel("ERROR")

## **Part 1: Creating Streaming Data Using `rate`**

The code below sets up a data stream using the `rate` format.

In [4]:
rateDF = spark.readStream.format("rate").load()

The code below sets up a sequence of actions using appropriate functions from `pyspark.sql.functions` that uses the `rate` data to:
- find the square root of the rate value
- find mod 4 of the rate value

**Note:** This is done *prior* to starting the stream!

In [5]:
manipDF = rateDF.withColumn("sqrt_value", sqrt(col("value"))) \
                .withColumn("mod4_value", col("value") % 4)

We now output the above actions by creating a `writeStream` that writes to memory. The table is named `rate_sample`, and it can be accessed with `spark.sql()`. The query is started below.

In [6]:
writeDF = manipDF.writeStream.format("memory").queryName("rate_sample").start()

After about 30 seconds, we use the code below to stop the query.

In [7]:
writeDF.stop()

The code below outputs the entire table stored in the query name.

In [8]:
spark.sql("select * from rate_sample").show(30)

+--------------------+-----+------------------+----------+
|           timestamp|value|        sqrt_value|mod4_value|
+--------------------+-----+------------------+----------+
|2026-04-20 23:22:...|    0|               0.0|         0|
|2026-04-20 23:22:...|    1|               1.0|         1|
|2026-04-20 23:22:...|    2|1.4142135623730951|         2|
|2026-04-20 23:22:...|    3|1.7320508075688772|         3|
|2026-04-20 23:22:...|    4|               2.0|         0|
|2026-04-20 23:22:...|    5|  2.23606797749979|         1|
|2026-04-20 23:22:...|    6| 2.449489742783178|         2|
|2026-04-20 23:22:...|    7|2.6457513110645907|         3|
|2026-04-20 23:22:...|    8|2.8284271247461903|         0|
|2026-04-20 23:22:...|    9|               3.0|         1|
|2026-04-20 23:22:...|   10|3.1622776601683795|         2|
|2026-04-20 23:22:...|   11|   3.3166247903554|         3|
|2026-04-20 23:22:...|   12|3.4641016151377544|         0|
|2026-04-20 23:22:...|   13| 3.605551275463989|         

## **Part 2: Using Data from a CSV with a Pipeline**

For this part of the assignment, we are given six `bikeDetails` sub datasets. We start by reading in the file `bikeDetails_for_fit.csv` as a spark SQL dataframe. This dataframe will be called `bike_data`.

In [9]:
bike_data = spark.read.load("bikeDetails_for_fit.csv", format = "csv", sep = ",",
                            inferSchema = "true", header = "true")
bike_data.show(5)

+--------------------+-------------+----+-----------+---------+---------+-----------------+
|                name|selling_price|year|seller_type|    owner|km_driven|ex_showroom_price|
+--------------------+-------------+----+-----------+---------+---------+-----------------+
|Royal Enfield Cla...|       175000|2019| Individual|1st owner|      350|             NULL|
|           Honda Dio|        45000|2017| Individual|1st owner|     5650|             NULL|
|Royal Enfield Cla...|       150000|2018| Individual|1st owner|    12000|           148114|
|Yamaha Fazer FI V...|        65000|2015| Individual|1st owner|    23000|            89643|
|Yamaha SZ [2013-2...|        20000|2011| Individual|2nd owner|    21000|             NULL|
+--------------------+-------------+----+-----------+---------+---------+-----------------+
only showing top 5 rows


Next, we use an `SQLTransformer()` to run the following, which was provided in the assignment.

In [10]:
sqlTrans = SQLTransformer(
    statement = """
                SELECT log(selling_price) as label, year, log(km_driven) as log_km_driven,
                CASE WHEN owner = '1st owner' THEN 1 ELSE 0 END AS one_owner
                FROM __THIS__
                """
            )
sqlTrans.transform(bike_data).show(5)

+------------------+----+------------------+---------+
|             label|year|     log_km_driven|one_owner|
+------------------+----+------------------+---------+
|12.072541252905651|2019| 5.857933154483459|        1|
|10.714417768752456|2017| 8.639410824140487|        1|
|11.918390573078392|2018| 9.392661928770137|        1|
|11.082142548877775|2015|10.043249494911286|        1|
| 9.903487552536127|2011|  9.95227771670556|        0|
+------------------+----+------------------+---------+
only showing top 5 rows


Continuing on, we use a `VectorAssembler()` to create a `features` column, which will include the `year`,
`log_km_driven`, and `one_owner` variables.

In [11]:
assembler = VectorAssembler(
                inputCols = ["year", "log_km_driven", "one_owner"],
                outputCol = "features",
                handleInvalid = "keep"
             )

assembler.transform(
    sqlTrans.transform(bike_data)
).show(5)

+------------------+----+------------------+---------+--------------------+
|             label|year|     log_km_driven|one_owner|            features|
+------------------+----+------------------+---------+--------------------+
|12.072541252905651|2019| 5.857933154483459|        1|[2019.0,5.8579331...|
|10.714417768752456|2017| 8.639410824140487|        1|[2017.0,8.6394108...|
|11.918390573078392|2018| 9.392661928770137|        1|[2018.0,9.3926619...|
|11.082142548877775|2015|10.043249494911286|        1|[2015.0,10.043249...|
| 9.903487552536127|2011|  9.95227771670556|        0|[2011.0,9.9522777...|
+------------------+----+------------------+---------+--------------------+
only showing top 5 rows


Additionally, we create a `Pipeline` with the two steps above:
- `sqlTrans`, created from `SQLTransformer()`
- `assembler`, created from `VectorAssembler()`

**Note:** The `Pipeline` created is called `pipeline`!

In [12]:
pipeline = Pipeline(stages = [sqlTrans, assembler])

Finally, we fit the above `pipeline` to our SQL dataframe, `bike_data`. This will be saved as an object called `bike_fit`.

In [13]:
bike_fit = pipeline.fit(bike_data)

Now that we have our fitted pipeline, we want to set up a read stream to look for csv files placed into the `csv_files` folder. When a csv comes in, we want to transform it using the fitted pipeline’s `.transform()` method.

In [19]:
myschema = bike_data.schema
bike_stream = spark.readStream.option("header", "true").schema(myschema).csv("csv_files")

Now that our read stream is set up, we can set up the transformations for the csv's that come in using the fitted pipeline.

In [20]:
bike_transform = bike_fit.transform(bike_stream)

We can also start our query. After all `bikeDetails_add*.csv` files are dragged into the `csv_files` folder, we will stop the query.
**Note:** The `csv_files` folder must be empty before we start our query!

In [21]:
query = bike_transform.writeStream.outputMode("append").format("console").start()

-------------------------------------------
Batch: 0
-------------------------------------------
+------------------+----+------------------+---------+--------------------+
|             label|year|     log_km_driven|one_owner|            features|
+------------------+----+------------------+---------+--------------------+
| 8.987196820661973|2003|10.887436932884098|        1|[2003.0,10.887436...|
|11.156250521031495|2018| 9.615805480084347|        1|[2018.0,9.6158054...|
|10.819778284410283|2016| 8.987196820661973|        1|[2016.0,8.9871968...|
| 10.46310334047155|2015|10.582738627903963|        1|[2015.0,10.582738...|
| 9.903487552536127|2006|11.225243392518447|        1|[2006.0,11.225243...|
|10.819778284410283|2012|10.239959789157341|        1|[2012.0,10.239959...|
| 10.51867319162636|2008| 11.03488966402723|        1|[2008.0,11.034889...|
|11.141861783579396|2018| 9.392661928770137|        1|[2018.0,9.3926619...|
|10.239959789157341|2012| 10.81975828421028|        1|[2012.0,10.81

In [22]:
query.stop()